# 03 - Poz modeli karşılaştırması
MediaPipe Pose ve RTMPose'u aynı görüntülerde landmark hatası, güven, açı hatası ve çalışma süresiyle karşılaştırır.

In [ ]:
import sys, zipfile
from pathlib import Path
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    artifacts_root = Path('/content/drive/MyDrive/PhysioLoad_AI_Project')
    code_root = Path('/content/physioload-ai')
    if not (code_root / 'physioload_env.py').exists():
        code_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(artifacts_root / 'code' / 'physioload-source.zip') as package: package.extractall(code_root)
else:
    artifacts_root = None
    here = Path.cwd().resolve()
    code_root = next(p for p in (here, *here.parents) if (p / 'physioload_env.py').exists())
sys.path.insert(0, str(code_root))
from physioload_env import setup_environment
env = setup_environment(root=code_root, artifacts_root=artifacts_root)
print(env.summary())

In [ ]:
import cv2, pandas as pd
from physioload.pose import MediaPipePoseEstimator, RTMPoseEstimator

backends = {}
for name, factory in [('mediapipe', MediaPipePoseEstimator), ('rtmpose', RTMPoseEstimator)]:
    try:
        backends[name] = factory()
    except Exception as error:
        print(name, 'kullanılamadı:', error)
print('hazır backends:', list(backends))

In [ ]:
def benchmark_image(image_path):
    bgr = cv2.imread(str(image_path))
    if bgr is None:
        raise FileNotFoundError(image_path)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    rows = []
    for name, estimator in backends.items():
        result = estimator.predict(rgb)
        rows.append({'backend': name, 'landmarks': len(result.names), 'mean_confidence': result.confidence.mean(), 'inference_ms': result.inference_ms})
    return pd.DataFrame(rows)

print('benchmark_image(path) hazır. Referans eşlemesiyle PCK ve açı MAE eklenmelidir.')

## Zorunlu kayıtlar
Her satır veri seti, kişi, deneme, kamera, görüntü, backend, model sürümü, işlenmiş veri sürümü ve runtime bilgisiyle kaydedilmelidir.